In [1]:
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)
print(app.llm_config)
print(type(app.adapter).__name__)

{'backend': 'ollama', 'gguf_model_path': 'gemma-2-9b-it-Q4_K_M.gguf', 'ollama_model_name': 'gemma3:4b', 'ollama_base_url': 'http://localhost:11434', 'n_ctx': 2048, 'n_gpu_layers': -1, 'max_tokens': 200, 'temperature': 0.1}
OllamaAdapter


In [2]:
from intake_engine.app_flow import IntakeAppFlow
from pprint import pprint

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "integration smoke test",
    primary_complaint = "headache",
    auto_save = False,
)

step = app.start_intake(auto_save = False)
pprint(step)

{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}


In [3]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "headache smoke test",
    primary_complaint = "headache",
    auto_save = False,
)

answers = [
    "I have had a headache since this morning.",
    "No, it did not start suddenly and severely.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No vision changes.",
    "No confusion.",
    "No fever or neck stiffness.",
    "No recent head trauma.",
    "No, I am not pregnant and not postpartum.",
    "This morning.",
    "About four hours.",
    "7 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "In the front of my head.",
    "Throbbing.",
]

pprint(app.start_intake(auto_save = False))

for answer in answers:
    result = app.submit_answer(answer, auto_save = False)
    pprint(result)

state = app.get_state()
print("\nFINAL HPI")
pprint(state["hpi"])

print("\nMISSING CLARIFICATIONS")
pprint(state["missing_clarifications"])

{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'headache',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_sudden_severe_onset',
 'next_question': 'Did the headache start suddenly and become very severe '
                  'right away?',
 'next_target': 'sudden_sev

In [4]:
from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("headache")
print(policy.policy_name)
print(policy.critical_followups)
print(policy.must_characterize)
print(policy.wrap_up_rule)

headache
['sudden_severe_onset', 'neurologic_symptoms', 'visual_changes', 'confusion_or_ams', 'fever_or_neck_stiffness', 'head_trauma', 'pregnancy_or_postpartum_context']
['onset', 'duration', 'severity', 'timing', 'course', 'location', 'character', 'aggravating_factors', 'relieving_factors', 'associated_symptoms']
{'type': 'characterization_threshold', 'require_all_critical': True, 'required_characterization_targets': ['onset', 'duration', 'severity', 'location', 'character', 'associated_symptoms'], 'min_required_characterization_count': 4}


In [5]:
from pprint import pprint

print("CURRENT STATE BEFORE CONTINUATION")
pprint(app.get_state()["hpi"])
print()
print("MISSING CLARIFICATIONS")
pprint(app.get_state()["missing_clarifications"])
print("-" * 80)

continuation_answers = [
    "Nausea and sensitivity to light.",
    "Bright light makes it worse.",
    "Rest helps a little.",
]

for i, answer in enumerate(continuation_answers, start = 1):
    print(f"CONTINUATION STEP {i}")
    print("PATIENT ANSWER:", answer)
    print()

    result = app.submit_answer(
        patient_answer = answer,
        auto_save = False,
    )

    pprint(result)
    print()

    state = app.get_state()

    print("CURRENT HPI")
    pprint(state["hpi"])
    print()

    print("CURRENT POLICY ANSWERS")
    pprint(state["policy_answers"])
    print()

    print("MISSING CLARIFICATIONS")
    pprint(state["missing_clarifications"])
    print("-" * 80)

    if not state["missing_clarifications"]:
        print("All missing clarifications resolved.")
        break

CURRENT STATE BEFORE CONTINUATION
{'aggravating_factors': [],
 'associated_symptoms': [],
 'character': 'Throbbing.',
 'context': None,
 'course': 'It has stayed about the same.',
 'duration': 'about four hours.',
 'location': 'In the front of my head.',
 'onset': 'This morning.',
 'relieving_factors': [],
 'severity': '7/10',
 'summary': None,
 'timing': 'It comes and goes.'}

MISSING CLARIFICATIONS
['associated symptoms']
--------------------------------------------------------------------------------
CONTINUATION STEP 1
PATIENT ANSWER: Nausea and sensitivity to light.

EXTRACTOR FALLBACK: ValueError append_fields must be a dictionary
{'applied_update': {'append_fields': {'hpi.aggravating_factors': ['nausea',
                                                                  'sensitivity '
                                                                  'to light.']},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    

In [6]:
from pprint import pprint

result = app.submit_answer(
    patient_answer = "Nausea and sensitivity to light.",
    auto_save = False,
)

pprint(result)
print()
pprint(app.get_state()["hpi"])
print()
pprint(app.get_state()["policy_answers"])
print()
pprint(app.get_state()["missing_clarifications"])

EXTRACTOR FALLBACK: ValueError append_fields must be a dictionary
{'applied_update': {'append_fields': {'hpi.aggravating_factors': ['nausea',
                                                                  'sensitivity '
                                                                  'to light.']},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_aggravating_factors',
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_aggravating_factors',
 'next_question': 'Can you tell me what makes your headache worse?',
 'next_target': 'aggravating_factors'}

{'aggravating_factors': ['nausea',
                         'sensitivity to light.',
                         'bright light makes it worse.',
                         'rest helps a